In [1]:
# example of a dcgan on cifar10
from numpy import expand_dims
from numpy import zeros
from numpy import ones
from numpy import vstack
from numpy.random import randn
from numpy.random import randint
from tensorflow.keras.datasets.cifar10 import load_data
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Reshape
from tensorflow.keras.layers import Flatten
from tensorflow.keras.layers import Conv2D
from tensorflow.keras.layers import Conv2DTranspose
from tensorflow.keras.layers import LeakyReLU
from tensorflow.keras.layers import Dropout
from matplotlib import pyplot

# define the standalone discriminator model
def define_discriminator(in_shape=(32,32,3)):
	model = Sequential()
	# normal
	model.add(Conv2D(64, (3,3), padding='same', input_shape=in_shape))
	model.add(LeakyReLU(alpha=0.2))
	# downsample
	model.add(Conv2D(128, (3,3), strides=(2,2), padding='same'))
	model.add(LeakyReLU(alpha=0.2))
	# downsample
	model.add(Conv2D(128, (3,3), strides=(2,2), padding='same'))
	model.add(LeakyReLU(alpha=0.2))
	# downsample
	model.add(Conv2D(256, (3,3), strides=(2,2), padding='same'))
	model.add(LeakyReLU(alpha=0.2))
	# classifier
	model.add(Flatten())
	model.add(Dropout(0.4))
	model.add(Dense(1, activation='sigmoid'))
	# compile model
	opt = Adam(learning_rate=0.0002, beta_1=0.5)
	model.compile(loss='binary_crossentropy', optimizer=opt, metrics=['accuracy'])
	return model

# define the standalone generator model
def define_generator(latent_dim):
	model = Sequential()
	# foundation for 4x4 image
	n_nodes = 256 * 4 * 4
	model.add(Dense(n_nodes, input_dim=latent_dim))
	model.add(LeakyReLU(alpha=0.2))
	model.add(Reshape((4, 4, 256)))
	# upsample to 8x8
	model.add(Conv2DTranspose(128, (4,4), strides=(2,2), padding='same'))
	model.add(LeakyReLU(alpha=0.2))
	# upsample to 16x16
	model.add(Conv2DTranspose(128, (4,4), strides=(2,2), padding='same'))
	model.add(LeakyReLU(alpha=0.2))
	# upsample to 32x32
	model.add(Conv2DTranspose(128, (4,4), strides=(2,2), padding='same'))
	model.add(LeakyReLU(alpha=0.2))
	# output layer
	model.add(Conv2D(3, (3,3), activation='tanh', padding='same'))
	return model

# define the combined generator and discriminator model, for updating the generator
def define_gan(g_model, d_model):
	# make weights in the discriminator not trainable
	d_model.trainable = False
	# connect them
	model = Sequential()
	# add generator
	model.add(g_model)
	# add the discriminator
	model.add(d_model)
	# compile model
	opt = Adam(learning_rate=0.0002, beta_1=0.5)
	model.compile(loss='binary_crossentropy', optimizer=opt)
	return model

# load and prepare cifar10 training images
def load_real_samples():
	# load cifar10 dataset
	(trainX, _), (_, _) = load_data()
	# convert from unsigned ints to floats
	X = trainX.astype('float32')
	# scale from [0,255] to [-1,1]
	X = (X - 127.5) / 127.5
	return X

# select real samples
def generate_real_samples(dataset, n_samples):
	# choose random instances
	ix = randint(0, dataset.shape[0], n_samples)
	# retrieve selected images
	X = dataset[ix]
	# generate 'real' class labels (1)
	y = ones((n_samples, 1))
	return X, y

# generate points in latent space as input for the generator
def generate_latent_points(latent_dim, n_samples):
	# generate points in the latent space
	x_input = randn(latent_dim * n_samples)
	# reshape into a batch of inputs for the network
	x_input = x_input.reshape(n_samples, latent_dim)
	return x_input

# use the generator to generate n fake examples, with class labels
def generate_fake_samples(g_model, latent_dim, n_samples):
	# generate points in latent space
	x_input = generate_latent_points(latent_dim, n_samples)
	# predict outputs
	X = g_model.predict(x_input)
	# create 'fake' class labels (0)
	y = zeros((n_samples, 1))
	return X, y

# create and save a plot of generated images
def save_plot(examples, epoch, n=7):
	# scale from [-1,1] to [0,1]
	examples = (examples + 1) / 2.0
	# plot images
	for i in range(n * n):
		# define subplot
		pyplot.subplot(n, n, 1 + i)
		# turn off axis
		pyplot.axis('off')
		# plot raw pixel data
		pyplot.imshow(examples[i])
	# save plot to file
	filename = 'generated_plot_e%03d.png' % (epoch+1)
	pyplot.savefig(filename)
	pyplot.close()

# evaluate the discriminator, plot generated images, save generator model
def summarize_performance(epoch, g_model, d_model, dataset, latent_dim, n_samples=150):
	# prepare real samples
	X_real, y_real = generate_real_samples(dataset, n_samples)
	# evaluate discriminator on real examples
	_, acc_real = d_model.evaluate(X_real, y_real, verbose=0)
	# prepare fake examples
	x_fake, y_fake = generate_fake_samples(g_model, latent_dim, n_samples)
	# evaluate discriminator on fake examples
	_, acc_fake = d_model.evaluate(x_fake, y_fake, verbose=0)
	# summarize discriminator performance
	print('>Accuracy real: %.0f%%, fake: %.0f%%' % (acc_real*100, acc_fake*100))
	# save plot
	save_plot(x_fake, epoch)
	# save the generator model tile file
	filename = 'generator_model_%03d.h5' % (epoch+1)
	g_model.save(filename)

# train the generator and discriminator
def train(g_model, d_model, gan_model, dataset, latent_dim, n_epochs=200, n_batch=128):
	bat_per_epo = int(dataset.shape[0] / n_batch)
	half_batch = int(n_batch / 2)
	# manually enumerate epochs
	for i in range(n_epochs):
		# enumerate batches over the training set
		for j in range(bat_per_epo):
			# get randomly selected 'real' samples
			X_real, y_real = generate_real_samples(dataset, half_batch)
			d_model.trainable = True
			# update discriminator model weights
			d_loss1, _ = d_model.train_on_batch(X_real, y_real)
			# generate 'fake' examples
			X_fake, y_fake = generate_fake_samples(g_model, latent_dim, half_batch)
			# update discriminator model weights
			d_loss2, _ = d_model.train_on_batch(X_fake, y_fake)
			d_model.trainable = False
			# prepare points in latent space as input for the generator
			X_gan = generate_latent_points(latent_dim, n_batch)
			# create inverted labels for the fake samples
			y_gan = ones((n_batch, 1))
			# update the generator via the discriminator's error
			g_loss = gan_model.train_on_batch(X_gan, y_gan)
			# summarize loss on this batch
			print('>%d, %d/%d, d1=%.3f, d2=%.3f g=%.3f' %
				(i+1, j+1, bat_per_epo, d_loss1, d_loss2, g_loss))
		# evaluate the model performance, sometimes
		if (i+1) % 10 == 0:
			summarize_performance(i, g_model, d_model, dataset, latent_dim)

# size of the latent space
latent_dim = 100
# create the discriminator
d_model = define_discriminator()
# create the generator
g_model = define_generator(latent_dim)
# create the gan
gan_model = define_gan(g_model, d_model)
# load image data
dataset = load_real_samples()
# train model
train(g_model, d_model, gan_model, dataset, latent_dim)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/activations/leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step 
>1, 1/390, d1=0.671, d2=0.684 g=0.691
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
>1, 2/390, d1=0.655, d2=0.667 g=0.688
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
>1, 3/390, d1=0.639, d2=0.651 g=0.684
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
>1, 4/390, d1=0.622, d2=0.636 g=0.676
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
>1, 5/390, d1=0.602, d2=0.622 g=0.662
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step
>1, 6/390, d1=0.587, d2=0.613 g=0.646
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
>1, 7/390, d1=0.580, d2=0.609 g=0.633
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
>1, 8/390, d1=0.580, d2=0.602 g=0.629
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
>1, 9/390, d1=0.577, d2=0.590 g=0.637
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
>1, 10/390, d1=0.570, d2=0.575 g=0.657
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
>1, 11/390, d1=0.558, d2=0.559 g=0.684
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
>1, 12/390, d1=0.544, d2=0.543 g=0.708
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 20m

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
>11, 1/390, d1=0.536, d2=0.575 g=1.446
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
>11, 2/390, d1=0.557, d2=0.569 g=1.446
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
>11, 3/390, d1=0.576, d2=0.569 g=1.446
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
>11, 4/390, d1=0.575, d2=0.571 g=1.445
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
>11, 5/390, d1=0.575, d2=0.574 g=1.445
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
>11, 6/390, d1=0.577, d2=0.575 g=1.445
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
>11, 7/390, d1=0.577, d2=0.575 g=1.445
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
>11, 8/390, d1=0.572, d2=0.566 g=1.445
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
>11, 9/390, d1=0.572, d2=0.566 g=1.445
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
>11, 10/390, d1=0.567, d2=0.567 g=1.445
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
>11, 11/390, d1=0.562, d2=0.560 g=1.445
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
>11, 12/390, d1=0.559, d2=0.557 g=1.445
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
>11, 13/390, d1=0.559, d2=0.560 g=1

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
>21, 1/390, d1=0.648, d2=0.668 g=1.188
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
>21, 2/390, d1=0.655, d2=0.657 g=1.188
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
>21, 3/390, d1=0.653, d2=0.657 g=1.188
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
>21, 4/390, d1=0.656, d2=0.664 g=1.188
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
>21, 5/390, d1=0.661, d2=0.661 g=1.188
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
>21, 6/390, d1=0.657, d2=0.658 g=1.188
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
>21, 7/390, d1=0.657, d2=0.655 g=1.188
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
>21, 8/390, d1=0.655, d2=0.651 g=1.188
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
>21, 9/390, d1=0.654, d2=0.653 g=1.188
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
>21, 10/390, d1=0.653, d2=0.652 g=1.188
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
>21, 11/390, d1=0.654, d2=0.652 g=1.188
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
>21, 12/390, d1=0.652, d2=0.650 g=1.188
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
>21, 13/390, d1=0.651, d2=0.651 g=1

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
>31, 1/390, d1=0.614, d2=0.632 g=1.077
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
>31, 2/390, d1=0.621, d2=0.634 g=1.077
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
>31, 3/390, d1=0.629, d2=0.636 g=1.077
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
>31, 4/390, d1=0.633, d2=0.636 g=1.077
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
>31, 5/390, d1=0.637, d2=0.644 g=1.077
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
>31, 6/390, d1=0.649, d2=0.652 g=1.077
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
>31, 7/390, d1=0.658, d2=0.651 g=1.077
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
>31, 8/390, d1=0.660, d2=0.653 g=1.077
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
>31, 9/390, d1=0.654, d2=0.655 g=1.077
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
>31, 10/390, d1=0.658, d2=0.653 g=1.077
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
>31, 11/390, d1=0.657, d2=0.657 g=1.077
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
>31, 12/390, d1=0.660, d2=0.657 g=1.077
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
>31, 13/390, d1=0.657, d2=0.657 g=1

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
>41, 1/390, d1=0.624, d2=0.635 g=1.018
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step
>41, 2/390, d1=0.651, d2=0.648 g=1.018
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
>41, 3/390, d1=0.651, d2=0.654 g=1.018
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
>41, 4/390, d1=0.653, d2=0.655 g=1.018
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
>41, 5/390, d1=0.656, d2=0.668 g=1.018
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
>41, 6/390, d1=0.670, d2=0.671 g=1.018
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
>41, 7/390, d1=0.668, d2=0.663 g=1.018
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
>41, 8/390, d1=0.663, d2=0.660 g=1.018
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
>41, 9/390, d1=0.661, d2=0.659 g=1.018
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
>41, 10/390, d1=0.657, d2=0.657 g=1.018
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
>41, 11/390, d1=0.655, d2=0.658 g=1.018
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
>41, 12/390, d1=0.658, d2=0.660 g=1.018
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
>41, 13/390, d1=0.658, d2=0.661 g=1

: 